In [ ]:
import numpy as np 
import pandas as pd
import statsmodels.api as sm
import statsmodels
import matplotlib.pyplot as plt

from patsy import dmatrices

import torch

# statsmodels introduction 

In [ ]:
df = sm.datasets.get_rdataset("Guerry", "HistData").data
df

In [ ]:
# interests vars
vars = ['Department', 'Lottery', 'Literacy', 'Wealth', 'Region']

df = df.dropna()

In [ ]:
##############################
# We use patsy to divide data data into, variables which gonna be descriptors and describers ~ R syntax 
# Important here is that it also adds constants and automatically use onehot auto encoders etc (to much for me in one file) also with appropriate naming variables !!!
#
##################################
y, X = dmatrices('Lottery ~ Literacy + Wealth + Region', data=df, return_type='dataframe')
X

In [ ]:
# model training 

# creating model object
model = sm.OLS(y, X)
res = model.fit() # we train model here (although we assigned data - it is because we can also do data manipulation here)

res.summary() # method which return model raport

In [ ]:
# we can also get only certain informations

res.rsquared # etc 
res.params
dir(res) # all attributes

# Testy Walda i Rao na iloraz wiarygodności 

In [ ]:
# stworzenie modelu
norm = torch.distributions.Normal(0, 1)
norm = norm.expand([20])
X = norm.sample()
y = 0.1 * X + norm.sample()

lm_full = sm.OLS(y.reshape([-1, 1]).numpy(), sm.add_constant(X.reshape([-1, 1]).numpy())).fit()
lm_null = sm.OLS(y.reshape([-1, 1]).numpy(), np.ones((20, 1))).fit()

In [ ]:
# Predykcje do rysowania
x_vals = np.linspace(X.min().item(), X.max().item(), 100)
x_vals_const = sm.add_constant(x_vals)
y_pred_full = lm_full.predict(x_vals_const)
y_pred_null = lm_null.predict(np.ones((100,1)))

# Wykres
plt.figure(figsize=(8,6))
plt.scatter(X.numpy(), y.numpy(), color='black', label='Dane')
plt.plot(x_vals, y_pred_full, color='blue', label='Model pełny: y ~ X')
plt.plot(x_vals, y_pred_null, color='red', linestyle='--', label='Model zerowy: y ~ 1')
plt.xlabel("X")
plt.ylabel("y")
plt.title("Porównanie modeli: pełny vs zerowy")
plt.legend()
plt.show()

## Testy
# sprawdzenie czynnika zerowego
print("Sprawdzenie czy współczynniki są równe zero\n", lm_full.wald_test("x1=0"))
# sprawdzenie modelu pełnego oraz zerowego
print('Pełny model - wald\n', lm_full.compare_lr_test(lm_null))


In [ ]:
# 